In [0]:
%sql
select *, row_number() over(partition by c2 order by c3 desc ) as rn from table 
-- window function -->row preservation + calculation

In [0]:
# %sql
# select c1 , c2 , count(*) from table 
# group by c1,c2  # row reduction 
# having count(*) >1 

In [0]:
from pyspark.sql.window import Window
lap_times_df = spark.table("formula1_dev.silver.lap_times")
lap_times_df.display()

In [0]:
from pyspark.sql.functions import col

In [0]:
from pyspark.sql.functions import col,row_number
dep_window= Window.partitionBy("lap").orderBy(col("milliseconds"))
lap_df= lap_times_df.withColumn("rn", row_number().over(dep_window))\
        .filter(col("rn")==2)
lap_df.display()

In [0]:
from pyspark.sql.functions import col,rank
dep_window= Window.partitionBy("lap").orderBy(col("milliseconds"))
lap_df_rank= lap_times_df.withColumn("rnk", rank().over(dep_window))\
        .filter(col("rnk")==2)
lap_df_rank.display()

In [0]:
# 10000 -->3 -->1,1,1
# 9000 --> 4

In [0]:
from pyspark.sql.functions import avg
driver_df= lap_times_df.groupBy("driver_id").agg(avg("milliseconds"))
# driver_df.display()
rank_window = Window.orderBy(col("avg(milliseconds)").desc())
driver_df1= driver_df.withColumn("lap_position_category", rank().over(rank_window))
driver_df1.display()

In [0]:
from pyspark.sql.functions import col,rank,dense_rank
dep_window= Window.partitionBy("lap").orderBy(col("milliseconds"))
lap_df_rank= lap_times_df.withColumn("rnk", rank().over(dep_window))\
        .filter(col("rnk")==2)
lap_df_rank.display()

In [0]:
employee = [("James","","Smith","36636","M",3000),
    ("Michael","Rose","","40288","M",4000),
    ("Robert","","Williams","42114","M",4000),
    ("Maria","Anne","Jones","39192","F",2000)]
schema = ["firstname","middlename","lastname","id","gender","salary"]
df = spark.createDataFrame(data=employee, schema = schema)
# df.display()
from pyspark.sql.functions import col,rank,dense_rank,row_number
df_row = df.withColumn("rn", row_number().over(Window.orderBy(col("salary").desc())))\
    .withColumn("rnk", rank().over(Window.orderBy(col("salary").desc())))\
    .withColumn("drnk", dense_rank().over(Window.orderBy(col("salary").desc())))
df_row.display()

In [0]:
# lap_times_df.show(20)
from pyspark.sql.functions import lag,lead
df_lag =lap_times_df.withColumn("lag_time", lag(col("milliseconds")).over(Window.partitionBy("driver_id").orderBy("race_id","lap")))\
    .withColumn("lead_time", lead(col("milliseconds")).over(Window.partitionBy("driver_id").orderBy("race_id","lap")))
df_lag.display()